<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-06b-fernwood-article-router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6, track (b) — Fernwood article router
**Course 1: Hands-On Deep Learning with Python — Chapter 6: Sequence models**

**Problem brief (Sam Okafor, Fernwood Media):** "Incoming wire articles need to be routed
to the right desk automatically." Target: macro-F1 ≥ 0.90 on AG News.

*(Choose this track OR track (a), `lab-06a-kestrel-hourly-forecaster.ipynb` — not both.)*

**What you'll submit:** a BiLSTM + attention classifier with a stratified split, the
macro-F1 result, and an error analysis.

In [ ]:
!pip install -q datasets

## 1. Load the data (with offline fallback)

In [ ]:
import numpy as np

np.random.seed(0)
LABEL_NAMES = ['World', 'Sports', 'Business', 'Sci/Tech']  # AG News's 4 desks

def load_ag_news(n_per_split=6000):
    try:
        from datasets import load_dataset
        ds = load_dataset('ag_news')
        train_texts = ds['train']['text'][:n_per_split]
        train_labels = ds['train']['label'][:n_per_split]
        val_texts = ds['test']['text'][:1000]
        val_labels = ds['test']['label'][:1000]
        print(f'Loaded the real AG News dataset: {len(train_texts)} train, {len(val_texts)} val.')
        return train_texts, train_labels, val_texts, val_labels
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — a tiny synthetic 4-class text set,')
        print('for pipeline development only (not a substitute for the real accuracy target).')
        templates = {
            0: ['Diplomats meet to discuss the {n} crisis', 'Elections held amid {n} tension'],
            1: ['The {n} team won the championship', 'Star player scores {n} points in the final'],
            2: ['Stock prices rose after the {n} earnings report', 'The central bank raised rates by {n} points'],
            3: ['Researchers unveil a new {n}-qubit processor', 'The company launched a {n}th-generation chip'],
        }
        def gen(n):
            texts, labels = [], []
            for i in range(n):
                label = i % 4
                t = np.random.choice(templates[label]).format(n=np.random.randint(1, 99))
                texts.append(t); labels.append(label)
            return texts, labels
        return (*gen(800), *gen(200))

train_texts, train_labels, val_texts, val_labels = load_ag_news()

## 2. A minimal tokenizer + vocabulary

In [ ]:
import re
from collections import Counter

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

counter = Counter(tok for t in train_texts for tok in tokenize(t))
VOCAB_SIZE = 10000
vocab = {'<pad>': 0, '<unk>': 1}
for tok, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[tok] = len(vocab)

def encode(text, max_len=40):
    ids = [vocab.get(tok, 1) for tok in tokenize(text)][:max_len]
    ids += [0] * (max_len - len(ids))
    return ids

MAX_LEN = 40
X_train = np.array([encode(t, MAX_LEN) for t in train_texts])
X_val = np.array([encode(t, MAX_LEN) for t in val_texts])
y_train = np.array(train_labels)
y_val = np.array(val_labels)
print('vocab size:', len(vocab), 'train:', X_train.shape, 'val:', X_val.shape)

## 3. BiLSTM + additive attention

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class AttentionRouter(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=64, n_classes=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bilstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.attn_score = nn.Linear(hidden * 2, 1)  # additive (Bahdanau-style) attention score
        self.classifier = nn.Linear(hidden * 2, n_classes)

    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)         # (batch, seq, 1)
        emb = self.embed(x)                             # (batch, seq, embed_dim)
        h, _ = self.bilstm(emb)                          # (batch, seq, 2*hidden)
        scores = self.attn_score(h).masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=1)            # (batch, seq, 1)
        context = (weights * h).sum(dim=1)                # (batch, 2*hidden) — attention-weighted sum
        return self.classifier(context), weights.squeeze(-1)


model = AttentionRouter(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.long)),
    batch_size=64, shuffle=True,
)

for epoch in range(8):
    model.train()
    epoch_loss, nb = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item(); nb += 1
    print(f'epoch {epoch}: train loss = {epoch_loss / nb:.4f}')

## 4. Evaluate: macro-F1

In [ ]:
from sklearn.metrics import f1_score, classification_report

model.eval()
with torch.no_grad():
    logits, attn_weights = model(torch.tensor(X_val, dtype=torch.long).to(device))
    val_preds = logits.argmax(1).cpu().numpy()

macro_f1 = f1_score(y_val, val_preds, average='macro')
print(f'Macro-F1: {macro_f1:.4f}  (target: >= 0.90 — on the real AG News set; the offline')
print('fallback set is far too small/simple to hit that, and that is expected.)')
print(classification_report(y_val, val_preds, target_names=LABEL_NAMES[:len(set(y_val))]))

## 5. What did each article's attention focus on?

In [ ]:
inv_vocab = {v: k for k, v in vocab.items()}

for i in range(3):
    toks = [inv_vocab.get(t, '<pad>') for t in X_val[i] if t != 0]
    w = attn_weights[i, :len(toks)].cpu().numpy()
    top = np.argsort(-w)[:5]
    print(f'Article: "{val_texts[i][:80]}..."')
    print('Top-attended tokens:', [(toks[j], round(float(w[j]), 3)) for j in top if j < len(toks)])
    print()

## 6. Error analysis (fill in)
Pull a few misclassified articles from `val_preds != y_val`. What do the errors have in
common — ambiguous topic overlap (e.g. sports business deals), short articles, something else?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 6: Sequence models*